# Exercise B: Bulk Data Processing at Scale

**Estimated time:** ~30 minutes

In research workflows, you frequently iterate on data processing pipelines: generate features, train models, evaluate, modify, retrain. When each iteration takes minutes on a local machine, the cumulative time cost slows down your research.

On Jetstream2 with 8+ CPU cores and 30+ GB RAM, the same pipeline runs faster — and the difference compounds over days of active research.

This exercise demonstrates:
- How CPU core count and memory affect data pipeline performance
- Feature engineering, PCA, and cross-validated model training on a 500K-row dataset
- Hyperparameter grid search — the kind of operation that bottlenecks local machines
- **Timing differences between your local machine and Jetstream2**

In [ ]:
!pip install pandas numpy scikit-learn scipy --quiet

import time
import os
import platform
import multiprocessing
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.metrics import classification_report


class timed:
    """Context manager for timing code blocks."""

    def __init__(self, label):
        self.label = label

    def __enter__(self):
        self.start = time.perf_counter()
        return self

    def __exit__(self, *args):
        self.elapsed = time.perf_counter() - self.start
        print(f"[{self.label}] {self.elapsed:.2f} seconds")

In [ ]:
print("=== System Information ===")
print(f"Platform:    {platform.platform()}")
print(f"CPU cores:   {multiprocessing.cpu_count()}")

# Try to get RAM info
try:
    with open("/proc/meminfo") as f:
        for line in f:
            if "MemTotal" in line:
                kb = int(line.split()[1])
                print(f"Total RAM:   {kb / (1024 ** 2):.1f} GB")
                break
except FileNotFoundError:
    try:
        import psutil
        print(f"Total RAM:   {psutil.virtual_memory().total / (1024 ** 3):.1f} GB")
    except ImportError:
        print("Total RAM:   (install psutil or run on Linux for RAM info)")

print(f"\nNote: Jetstream2 m3.medium has 8 CPUs and 30 GB RAM.")
print(f"      Compare the numbers above to see the difference.")

## Step 1: Generate a Synthetic Dataset

We generate a 500,000-row dataset to simulate a realistic research workload — sensor readings, categorical variables, and a binary classification target. Using synthetic data avoids download times and ensures consistent benchmarking across machines.

In [ ]:
N = 500_000
np.random.seed(42)

with timed("Generate 500K-row dataset"):
    # 40 numeric features (simulate sensor readings)
    numeric_data = np.random.randn(N, 40)
    df = pd.DataFrame(numeric_data, columns=[f"sensor_{i}" for i in range(40)])

    # 10 categorical features (encoded as integers)
    for i in range(10):
        df[f"category_{i}"] = np.random.randint(0, 5, N)

    # Binary target based on a combination of first 5 features
    df["target"] = (df.iloc[:, :5].sum(axis=1) > 0).astype(int)

print(f"Dataset shape: {df.shape}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024 ** 2:.1f} MB")
print(f"Target distribution:\n{df['target'].value_counts()}")
df.head()

## Step 2: Feature Engineering

The first bottleneck in any data pipeline is feature engineering. Rolling statistics, interaction terms, and group-by aggregations are CPU and memory intensive. More cores and memory allow these operations to complete faster.

In [ ]:
with timed("Feature engineering"):
    # Rolling statistics on first 10 numeric features
    for col in [f"sensor_{i}" for i in range(10)]:
        df[f"{col}_rolling_mean"] = df[col].rolling(window=100, min_periods=1).mean()
        df[f"{col}_rolling_std"] = df[col].rolling(window=100, min_periods=1).std()

    # Pairwise interactions (first 5 features)
    for i in range(5):
        for j in range(i + 1, 5):
            df[f"interact_{i}_{j}"] = df[f"sensor_{i}"] * df[f"sensor_{j}"]

    # Group-by aggregations
    for cat_col in [f"category_{i}" for i in range(3)]:
        for stat_col in [f"sensor_{i}" for i in range(5)]:
            group_mean = df.groupby(cat_col)[stat_col].transform("mean")
            df[f"{stat_col}_by_{cat_col}_mean"] = group_mean

print(f"Expanded dataset shape: {df.shape}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024 ** 2:.1f} MB")

## Step 3: Scaling and Dimensionality Reduction

In [ ]:
# Separate features and target
feature_cols = [c for c in df.columns if c != "target"]
X = df[feature_cols].fillna(0).values
y = df["target"].values

with timed("StandardScaler fit + transform"):
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

with timed("PCA (50 components)"):
    pca = PCA(n_components=50, random_state=42)
    X_pca = pca.fit_transform(X_scaled)

print(f"PCA output shape: {X_pca.shape}")
print(f"Variance explained: {pca.explained_variance_ratio_.sum():.2%}")

## Step 4: Model Training with Cross-Validation

Cross-validation with `n_jobs=-1` uses **all available CPU cores**. This is where the core count difference between your laptop and Jetstream2 shows up most clearly.

In [ ]:
print(f"Using all {multiprocessing.cpu_count()} available cores\n")

with timed("5-fold CV, RandomForest (n_jobs=-1, using all cores)"):
    rf_scores = cross_val_score(
        RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42),
        X_pca, y, cv=5, scoring="accuracy", n_jobs=-1,
    )

print(f"RandomForest accuracy: {rf_scores.mean():.4f} (+/- {rf_scores.std():.4f})")

## Step 5: Hyperparameter Grid Search

Grid search multiplies the cost: each parameter combination is evaluated across each CV fold. This is the kind of operation that makes researchers want more compute.

In [ ]:
param_grid = {
    "n_estimators": [50, 200],
    "max_depth": [10, 30],
    "min_samples_split": [2, 10],
}

print(f"Grid search: {2 * 2 * 2} parameter combinations x 3 CV folds = 24 model fits")
print(f"Using all {multiprocessing.cpu_count()} available cores\n")

with timed("GridSearchCV (8 combos x 3-fold)"):
    grid = GridSearchCV(
        RandomForestClassifier(n_jobs=-1, random_state=42),
        param_grid,
        cv=3,
        scoring="accuracy",
        n_jobs=-1,
        verbose=0,
    )
    grid.fit(X_pca, y)

print(f"\nBest parameters: {grid.best_params_}")
print(f"Best CV accuracy: {grid.best_score_:.4f}")

## Timing Comparison

| Operation                          | Local (your laptop) | Jetstream2 (m3.medium) |
|------------------------------------|--------------------:|-----------------------:|
| Generate 500K-row dataset          |          ___ sec    |             ___ sec    |
| Feature engineering                |          ___ sec    |             ___ sec    |
| StandardScaler                     |          ___ sec    |             ___ sec    |
| PCA (50 components)                |          ___ sec    |             ___ sec    |
| 5-fold CV (RandomForest)           |          ___ sec    |             ___ sec    |
| GridSearchCV (8x3)                 |          ___ sec    |             ___ sec    |
| **Total**                          |      **___ sec**    |         **___ sec**    |

**Fill in both columns.** The GridSearchCV row typically shows the largest difference because it fully utilizes parallel cores.

## The Research Iteration Loop

In a real research workflow, you iterate: try a feature set, train, evaluate, modify, retrain. If each iteration takes 5 minutes locally but 1 minute on Jetstream2, your 20-iteration experiment goes from **100 minutes to 20 minutes**.

Over a week of active research, this difference compounds into **hours saved**.

The point is not just raw speed — it is the ability to explore more hypotheses, try more configurations, and iterate faster on your research questions.

## Try It Yourself

- Try increasing `N` to 1,000,000 rows — how do the times change?
- Try adding more complex feature engineering
- Try `GradientBoostingClassifier` (sequential — benefits less from multiple cores)
- What is the smallest dataset size where you notice a difference between local and Jetstream2?

In [ ]:
# GradientBoosting is sequential (no n_jobs) -- how does it compare?
with timed("5-fold CV, GradientBoosting (sequential, 50K subset)"):
    gb_scores = cross_val_score(
        GradientBoostingClassifier(n_estimators=50, max_depth=5, random_state=42),
        X_pca[:50000], y[:50000],
        cv=5, scoring="accuracy",
    )

print(f"GradientBoosting accuracy (50K subset): {gb_scores.mean():.4f} (+/- {gb_scores.std():.4f})")
print(f"\nNote: GradientBoosting is sequential -- it benefits less from multiple cores")
print(f"but still benefits from faster memory and CPU clock speed on Jetstream2.")